# 10 ACE-Step: three score candidates
Settings: GPU on, Internet on. Add the dataset `heritage-short-bundle` (from `make kaggle-bundle`). Download `/kaggle/working/music/` and `notebook_log.jsonl` when it finishes.

In [ ]:
!pip install -q uv
!git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git /kaggle/working/ace
!cd /kaggle/working/ace && uv sync

In [ ]:
%%writefile /kaggle/working/gen_music.py
import json, os, time
from pathlib import Path
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

BUNDLE = Path("/kaggle/input/heritage-short-bundle")
OUT = Path("/kaggle/working/music")
OUT.mkdir(exist_ok=True)
LOG = Path("/kaggle/working/notebook_log.jsonl")
DONE = OUT / "done.json"
if DONE.exists():
    print("candidates already generated, nothing to do")
    raise SystemExit

music = json.loads((BUNDLE / "storyboard.json").read_text())["music"]
dit = AceStepHandler()
llm = LLMHandler()
root = os.getcwd()
dit.initialize_service(project_root=root, config_path="acestep-v15-turbo", device="cuda")
for backend in ("vllm", "pt"):
    try:
        llm.initialize(checkpoint_dir=os.path.join(root, "checkpoints"), lm_model_path="acestep-5Hz-lm-0.6B",
                       backend=backend, device="cuda")
        break
    except Exception as e:
        print("LLM backend", backend, "failed:", e)

params = GenerationParams(caption=music["caption"], bpm=music["bpm"], keyscale=music["key"],
                          duration=music["duration_s"], instrumental=True, seed=-1, lyrics="")
config = GenerationConfig(batch_size=3, audio_format="flac")
start = time.time()
result = generate_music(dit, llm, params, config, save_dir=str(OUT))
seconds = time.time() - start
if not result.success:
    raise SystemExit(f"ACE-Step failed: {result.error}")
paths = [a["path"] for a in result.audios]
with LOG.open("a") as f:
    f.write(json.dumps({"stage": "music", "model": "ace-step-1.5-turbo", "seconds": round(seconds, 1),
                        "candidates": paths, "seeds": [a["params"]["seed"] for a in result.audios]}) + "\n")
DONE.write_text(json.dumps(paths))
print("done in", round(seconds), "s:", paths)

In [ ]:
!cd /kaggle/working/ace && uv run python /kaggle/working/gen_music.py